# 🏭 Explainable Multi-Class Industrial Defect Detection
## Using Vision Transformers and Dual-Layer Explainability (Grad-CAM + Attention Fusion)

### 🎯 Project Overview
This project implements an explainable AI system for industrial defect detection that:
- **Detects 6 types of surface defects** using Vision Transformers
- **Explains predictions** with dual-layer visualization (Grad-CAM + Attention Maps)
- **Generates root-cause explanations** in natural language
- **Provides interactive GUI** for real-time defect analysis

### 📊 Dataset: NEU Surface Defect Database
- **1,800 grayscale images** (300×300px) of steel surfaces
- **6 defect classes**: Crazing, Inclusion, Patches, Pitted Surface, Rolled-in Scale, Scratches
- **Goal**: >92% accuracy with full explainability

---

## 🔧 Section 1: Environment Setup and Library Imports

Installing and importing all required libraries for the complete pipeline.

In [ ]:
# Install required packages (run once)
!pip install torch torchvision transformers
!pip install opencv-python matplotlib seaborn plotly
!pip install streamlit gradio kaggle
!pip install scikit-learn pillow numpy pandas
!pip install grad-cam pytorch-grad-cam
!pip install datasets

print("✅ All packages installed successfully!")

In [ ]:
# Core Libraries
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Data Processing
import numpy as np
import pandas as pd
from PIL import Image
import cv2

# Deep Learning
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from transformers import ViTForImageClassification, ViTImageProcessor, ViTModel
import torch.nn.functional as F

# Explainability
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Evaluation
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split

# Streamlit for deployment
import streamlit as st

# Utilities
import json
import random
from pathlib import Path
from tqdm.auto import tqdm

print("📦 All libraries imported successfully!")
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"🔥 CUDA available: {torch.cuda.is_available()}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

## 📊 Section 2: Dataset Preparation and Exploration

Setting up the NEU Surface Defect Dataset and exploring the data distribution.

In [ ]:
# Dataset Configuration
DATASET_PATH = "data"
CLASS_NAMES = ['Crazing', 'Inclusion', 'Patches', 'Pitted', 'Rolled', 'Scratches']
NUM_CLASSES = len(CLASS_NAMES)
IMAGE_SIZE = 224

# Create directory structure
os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("visualizations", exist_ok=True)

print("🗂️ Dataset Configuration:")
print(f"   📁 Path: {DATASET_PATH}")
print(f"   🏷️ Classes: {CLASS_NAMES}")
print(f"   🔢 Number of classes: {NUM_CLASSES}")
print(f"   📐 Image size: {IMAGE_SIZE}×{IMAGE_SIZE}")

# Defect explanations dictionary
DEFECT_EXPLANATIONS = {
    'Crazing': "Surface cracks caused by thermal stress or uneven cooling during the manufacturing process.",
    'Inclusion': "Non-metallic impurities embedded in the steel matrix during the rolling or casting process.",
    'Patches': "Irregular surface texture variations caused by inconsistent coating or material composition.",
    'Pitted': "Localized corrosion pits formed due to chemical contamination or environmental exposure (Pitted Surface).",
    'Rolled': "Oxide scale particles trapped and embedded in the surface during hot rolling operations (Rolled-in Scale).",
    'Scratches': "Linear mechanical abrasions caused by contact with tools, handling equipment, or processing machinery."
}

In [ ]:
# Verify Dataset Structure
def verify_dataset_structure():
    """Verify the organized dataset structure with train/valid/test splits"""
    
    print("🔍 Verifying dataset structure...")
    
    splits = ['train', 'valid', 'test']
    dataset_ok = True
    
    for split in splits:
        split_path = Path(DATASET_PATH) / split
        
        if not split_path.exists():
            print(f"⚠️ Split folder not found: {split}")
            dataset_ok = False
            continue
        
        print(f"\n📂 {split.upper()} Split:")
        split_total = 0
        
        for class_name in CLASS_NAMES:
            class_path = split_path / class_name
            
            if class_path.exists():
                # Count all image files
                image_files = list(class_path.glob("*.jpg")) + list(class_path.glob("*.png")) + \
                             list(class_path.glob("*.bmp")) + list(class_path.glob("*.jpeg"))
                
                num_files = len(image_files)
                split_total += num_files
                print(f"   ✅ {class_name}: {num_files} images")
            else:
                print(f"   ❌ {class_name}: folder not found")
                dataset_ok = False
        
        print(f"   📊 Total {split} images: {split_total}")
    
    if dataset_ok:
        print("\n✅ Dataset structure verified successfully!")
    else:
        print("\n⚠️ Dataset structure has missing folders")
    
    return dataset_ok

# Verify the dataset
dataset_ready = verify_dataset_structure()

In [ ]:
# Enhanced Dataset Analysis
def analyze_dataset():
    """Analyze the organized dataset with train/valid/test splits"""
    print("🔍 Performing Dataset Analysis...")
    
    # Use train split for visualization
    train_path = os.path.join(DATASET_PATH, 'train')
    
    if not os.path.exists(train_path):
        print(f"❌ Train folder not found at: {train_path}")
        return {}, False
    
    class_counts = {}
    total_images = 0
    
    # Count images in each class
    for class_name in CLASS_NAMES:
        class_path = os.path.join(train_path, class_name)
        if os.path.exists(class_path):
            image_files = [f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
            class_counts[class_name] = len(image_files)
            total_images += len(image_files)
        else:
            class_counts[class_name] = 0
    
    # Create visualization
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('🔍 Dataset Sample Visualization (Training Set)', fontsize=16, fontweight='bold')
    
    for idx, class_name in enumerate(CLASS_NAMES):
        row, col = idx // 3, idx % 3
        ax = axes[row, col]
        
        class_path = os.path.join(train_path, class_name)
        if os.path.exists(class_path):
            image_files = [f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
            if image_files:
                # Load and display first image
                img_path = os.path.join(class_path, image_files[0])
                img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
                
                ax.imshow(img, cmap='gray')
                
                # Calculate statistics
                mean_intensity = np.mean(img)
                std_intensity = np.std(img)
                
                title = f'{class_name}\n'
                title += f'{class_counts[class_name]} images\n'
                title += f'μ={mean_intensity:.1f}, σ={std_intensity:.1f}'
                
                ax.set_title(title, fontweight='bold', fontsize=11)
                ax.axis('off')
                
                # Add border
                for spine in ax.spines.values():
                    spine.set_edgecolor('green')
                    spine.set_linewidth(2)
        else:
            ax.text(0.5, 0.5, f'{class_name}\nNot Found', 
                   ha='center', va='center', fontsize=12, color='red')
            ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\n📊 Training Dataset Statistics:")
    print(f"   🖼️ Total images: {total_images}")
    print(f"   📦 Average per class: {total_images/6:.1f}")
    
    for class_name, count in class_counts.items():
        status = "✅" if count > 0 else "❌"
        print(f"   {status} {class_name}: {count} images")
    
    return class_counts, True

# Run dataset analysis
dataset_stats, is_real_dataset = analyze_dataset()

## 🔄 Section 3: Data Preprocessing and Augmentation

Implementing robust preprocessing pipeline and data augmentation strategies.

In [ ]:
# Custom Dataset Class
class DefectDataset(Dataset):
    """Custom PyTorch Dataset for NEU Surface Defect Classification"""
    
    def __init__(self, data_path, class_names, transform=None, mode='train'):
        self.data_path = data_path
        self.class_names = class_names
        self.transform = transform
        self.mode = mode
        
        # Create class to index mapping
        self.class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
        
        # Collect all image paths and labels
        self.image_paths = []
        self.labels = []
        
        for class_name in class_names:
            class_path = os.path.join(data_path, class_name)
            if os.path.exists(class_path):
                for filename in os.listdir(class_path):
                    if filename.endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                        self.image_paths.append(os.path.join(class_path, filename))
                        self.labels.append(self.class_to_idx[class_name])
        
        print(f"📊 {mode.capitalize()} Dataset: {len(self.image_paths)} images loaded")
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        # Load image
        img_path = self.image_paths[idx]
        image = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        
        # Convert to RGB (3 channels) for ViT
        image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        image = Image.fromarray(image)
        
        # Apply transforms
        if self.transform:
            image = self.transform(image)
        
        label = self.labels[idx]
        return image, label

# Data Transforms for Training and Validation
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("✅ Data transforms defined:")
print("   🔄 Training: Resize + Augmentation + Normalization")
print("   🔍 Validation: Resize + Normalization only")

In [ ]:
# Create data loaders from organized train/valid splits
def create_data_loaders():
    """Create training and validation data loaders from existing train/valid splits"""
    
    # Use the existing train and valid directories
    train_path = os.path.join(DATASET_PATH, 'train')
    valid_path = os.path.join(DATASET_PATH, 'valid')
    
    # Check if organized dataset exists
    if not os.path.exists(train_path) or not os.path.exists(valid_path):
        print(f"❌ Error: Train/valid directories not found!")
        print(f"   Expected: {train_path} and {valid_path}")
        print(f"   Please organize your dataset with train/valid/test splits")
        return None, None, 8
    
    # Create datasets from train and valid directories
    train_dataset = DefectDataset(train_path, CLASS_NAMES, transform=train_transforms, mode='train')
    val_dataset = DefectDataset(valid_path, CLASS_NAMES, transform=val_transforms, mode='validation')
    
    # Determine batch size based on dataset size
    total_samples = len(train_dataset) + len(val_dataset)
    if total_samples > 1000:  # Real NEU dataset
        batch_size = 16
        print("🏭 Large dataset detected - Using batch size 16")
    else:  # Smaller dataset
        batch_size = 8
        print("🔬 Small dataset detected - Using batch size 8")
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )
    val_loader = DataLoader(
        val_dataset, 
        batch_size=batch_size, 
        shuffle=False, 
        num_workers=0,
        pin_memory=torch.cuda.is_available()
    )
    
    print(f"\n📊 Data Loaders Created:")
    print(f"   🎯 Training samples: {len(train_dataset)}")
    print(f"   🔍 Validation samples: {len(val_dataset)}")
    print(f"   📦 Batch size: {batch_size}")
    
    # Show class distribution
    train_class_counts = {}
    val_class_counts = {}
    
    for label in train_dataset.labels:
        class_name = CLASS_NAMES[label]
        train_class_counts[class_name] = train_class_counts.get(class_name, 0) + 1
        
    for label in val_dataset.labels:
        class_name = CLASS_NAMES[label]
        val_class_counts[class_name] = val_class_counts.get(class_name, 0) + 1
    
    print(f"\n📈 Class Distribution:")
    for class_name in CLASS_NAMES:
        train_count = train_class_counts.get(class_name, 0)
        val_count = val_class_counts.get(class_name, 0)
        print(f"   {class_name}: Train={train_count}, Val={val_count}")
    
    return train_loader, val_loader, batch_size

# Create data loaders
train_loader, val_loader, BATCH_SIZE = create_data_loaders()

## 🤖 Section 4: Vision Transformer Model Implementation

Loading and configuring the pre-trained ViT model for defect classification.

In [ ]:
# Enhanced Vision Transformer for Defect Detection
class ExplainableViTDefectClassifier(nn.Module):
    """Enhanced ViT model with explainability features"""
    
    def __init__(self, num_classes=6, model_name="google/vit-base-patch16-224-in21k"):
        super().__init__()
        
        # Load pre-trained ViT model
        self.vit = ViTForImageClassification.from_pretrained(
            model_name,
            num_labels=num_classes,
            ignore_mismatched_sizes=True
        )
        
        # Store attention weights for explainability
        self.attention_weights = []
        self.register_hooks()
        
        print(f"🤖 Loaded ViT model: {model_name}")
        print(f"🎯 Number of classes: {num_classes}")
        
    def register_hooks(self):
        """Register hooks to capture attention weights"""
        def hook_fn(module, input, output):
            # Store attention weights for explainability
            if hasattr(output, 'attentions') and output.attentions is not None:
                self.attention_weights.append(output.attentions)
        
        # Register hook on the ViT model
        self.vit.vit.encoder.register_forward_hook(hook_fn)
    
    def forward(self, x):
        self.attention_weights = []  # Clear previous attention weights
        outputs = self.vit(x)
        return outputs
    
    def get_attention_maps(self):
        """Get the stored attention weights"""
        return self.attention_weights

# Initialize the model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ExplainableViTDefectClassifier(num_classes=NUM_CLASSES)
model = model.to(device)

print(f"🔥 Model initialized on: {device}")

# Model summary
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"📊 Model Parameters:")
print(f"   📈 Total parameters: {total_params:,}")
print(f"   🎯 Trainable parameters: {trainable_params:,}")

## 📥 Section 5: Load Pre-trained Model

Load the trained Vision Transformer model from the Colab training session.

**Note:** Train the model using `ExplainableDefectDetection_Colab.ipynb` first, then download the trained weights from Google Drive to the `models/` folder.

In [ ]:
# Load the trained model weights
MODEL_PATH = 'models/best_vit_defect_detector.pt'

if os.path.exists(MODEL_PATH):
    print(f"📥 Loading trained model from: {MODEL_PATH}")
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
    model.eval()
    print("✅ Model loaded successfully!")
    print(f"🔥 Model is on: {device}")
else:
    print(f"❌ Model file not found at: {MODEL_PATH}")
    print("📝 Please train the model using ExplainableDefectDetection_Colab.ipynb first")
    print("📥 Then download 'best_vit_defect_detector.pt' from Google Drive to the 'models/' folder")

## 🔍 Section 6: Grad-CAM Implementation

Implementing Gradient-based Class Activation Mapping for visual explanations.

In [ ]:
# Custom Grad-CAM for Vision Transformers
class ViTGradCAM:
    """Custom Grad-CAM implementation for Vision Transformers"""
    
    def __init__(self, model, target_layer=None):
        self.model = model
        self.gradients = None
        self.activations = None
        
        # Default to the last layer norm of the ViT encoder
        if target_layer is None:
            target_layer = self.model.vit.vit.layernorm
        
        self.target_layer = target_layer
        self.hook = target_layer.register_forward_hook(self.save_activation)
        self.gradient_hook = None  # Will register after forward pass
    
    def save_activation(self, module, input, output):
        """Save forward activations"""
        # output shape: [batch, seq_len, hidden_dim]
        self.activations = output
        
        # Register backward hook on the activations tensor
        if output.requires_grad:
            self.gradient_hook = output.register_hook(self.save_gradient)
    
    def save_gradient(self, grad):
        """Save backward gradients"""
        self.gradients = grad
    
    def generate_cam(self, input_tensor, target_class=None):
        """Generate CAM for given input"""
        # Forward pass
        self.model.eval()
        outputs = self.model(input_tensor)
        
        # Get prediction if target_class not specified
        if target_class is None:
            target_class = outputs.logits.argmax(dim=1).item()
        
        # Backward pass
        self.model.zero_grad()
        loss = outputs.logits[0, target_class]
        loss.backward()
        
        # Generate CAM
        if self.gradients is not None and self.activations is not None:
            # Handle tensor dimensions
            # activations shape: [batch, seq_len, hidden_dim]
            # gradients shape: [batch, seq_len, hidden_dim]
            activations = self.activations[0]  # [seq_len, hidden_dim]
            gradients = self.gradients[0]      # [seq_len, hidden_dim]
            
            # Global average pooling of gradients across the hidden dimension (features)
            # This gives us the importance weight for each spatial token
            weights = gradients.mean(dim=-1)  # [seq_len]
            
            # Also get the mean activation for each token
            activations_pooled = activations.mean(dim=-1)  # [seq_len]
            
            # Weighted combination: element-wise multiply
            cam = weights * activations_pooled  # [seq_len]
            
            # Apply ReLU and normalize
            cam = F.relu(cam)
            if cam.max() > cam.min():
                cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
            else:
                # If all values are the same, create uniform distribution
                cam = torch.ones_like(cam) * 0.5
            
            # Reshape to 2D (patch-based), excluding CLS token
            # For ViT-base-patch16-224: 14x14 = 196 patches + 1 CLS token = 197 tokens total
            cam_tokens = cam[1:]  # Exclude CLS token at position 0
            num_patches = cam_tokens.shape[0]
            patch_size = int(np.sqrt(num_patches))
            
            if patch_size * patch_size == num_patches:
                cam_2d = cam_tokens.reshape(patch_size, patch_size)
            else:
                # Fallback: try to create closest square
                patch_size = 14  # Default for ViT-base with 224x224 input
                if num_patches >= patch_size * patch_size:
                    cam_2d = cam_tokens[:patch_size*patch_size].reshape(patch_size, patch_size)
                else:
                    cam_2d = F.interpolate(
                        cam_tokens.unsqueeze(0).unsqueeze(0), 
                        size=(patch_size, patch_size), 
                        mode='bilinear'
                    ).squeeze()
            
            cam_np = cam_2d.detach().cpu().numpy()
            
            # Validate output
            if cam_np.size == 0:
                print(f"⚠️ Warning: CAM is empty. cam shape: {cam.shape}, num_patches: {num_patches}")
                return None, target_class
            
            return cam_np, target_class
        
        return None, target_class
    
    def cleanup(self):
        """Remove hooks"""
        self.hook.remove()

def apply_gradcam_to_image(model, image_tensor, original_image, target_class=None):
    """Apply Grad-CAM and overlay on original image"""
    
    # Initialize Grad-CAM
    gradcam = ViTGradCAM(model)
    
    try:
        # Generate CAM
        cam, predicted_class = gradcam.generate_cam(image_tensor.unsqueeze(0), target_class)
        
        if cam is not None and cam.size > 0:
            # Resize CAM to match input image
            cam_resized = cv2.resize(cam, (IMAGE_SIZE, IMAGE_SIZE))
            
            # Convert and resize original image for overlay
            if len(original_image.shape) == 3:
                img_for_overlay = original_image.copy()
            else:
                img_for_overlay = cv2.cvtColor(original_image, cv2.COLOR_GRAY2RGB)
            
            # Resize to IMAGE_SIZE to match cam_resized
            img_for_overlay = cv2.resize(img_for_overlay, (IMAGE_SIZE, IMAGE_SIZE))
            
            # Normalize image
            img_for_overlay = img_for_overlay.astype(np.float32) / 255.0
            
            # Create heatmap overlay
            heatmap = show_cam_on_image(img_for_overlay, cam_resized, use_rgb=True)
            
            return heatmap, cam_resized, predicted_class
        
    finally:
        gradcam.cleanup()
    
    return None, None, None

print("🔍 Grad-CAM implementation ready for ViT!")


## 👁️ Section 7: Attention Map Visualization

Extracting and visualizing ViT attention patterns for explainability.

In [ ]:
# ViT Attention Visualization
class ViTAttentionVisualizer:
    """Extract and visualize attention patterns from ViT"""
    
    def __init__(self, model):
        self.model = model
        self.attention_maps = []
        self.hooks = []
        self.register_hooks()
    
    def register_hooks(self):
        """Register hooks to capture attention weights"""
        def attention_hook(module, input, output):
            # Store attention weights from each transformer layer
            if hasattr(output, 'attentions') and output.attentions is not None:
                self.attention_maps.append(output.attentions.detach().cpu())
        
        # Register hooks on transformer layers
        for layer in self.model.vit.vit.encoder.layer:
            hook = layer.attention.register_forward_hook(attention_hook)
            self.hooks.append(hook)
    
    def get_attention_rollout(self, input_tensor, head_fusion='mean', layer_fusion='last'):
        """Compute attention rollout across layers and heads"""
        self.attention_maps = []
        
        # Forward pass to collect attention
        with torch.no_grad():
            _ = self.model(input_tensor)
        
        if not self.attention_maps:
            return None
        
        # Process attention maps
        attentions = []
        for attention in self.attention_maps:
            # attention shape: [batch, num_heads, seq_len, seq_len]
            if head_fusion == 'mean':
                attention_fused = attention.mean(dim=1)  # Average over heads
            elif head_fusion == 'max':
                attention_fused = attention.max(dim=1)[0]  # Max over heads
            else:
                attention_fused = attention[:, 0]  # First head
            
            attentions.append(attention_fused)
        
        # Rollout attention across layers
        rollout = torch.eye(attentions[0].size(-1)).unsqueeze(0)
        
        for attention in attentions:
            # Add residual connection and normalize
            attention = attention + torch.eye(attention.size(-1)).unsqueeze(0)
            attention = attention / attention.sum(dim=-1, keepdim=True)
            rollout = torch.matmul(rollout, attention)
        
        # Get attention for CLS token (index 0) or average
        if layer_fusion == 'cls':
            attention_map = rollout[0, 0, 1:]  # CLS to patch tokens
        else:
            attention_map = rollout[0].mean(dim=0)[1:]  # Average attention
        
        return attention_map
    
    def visualize_attention(self, input_tensor, original_image):
        """Create attention visualization overlay"""
        attention_weights = self.get_attention_rollout(input_tensor.unsqueeze(0))
        
        if attention_weights is None:
            return None, None
        
        # Reshape to 2D grid (patch-based)
        patch_size = int(np.sqrt(len(attention_weights)))
        attention_2d = attention_weights.reshape(patch_size, patch_size).numpy()
        
        # Resize to match input image
        attention_resized = cv2.resize(attention_2d, (IMAGE_SIZE, IMAGE_SIZE))
        attention_resized = (attention_resized - attention_resized.min()) / \
                           (attention_resized.max() - attention_resized.min() + 1e-8)
        
        # Convert original image for overlay
        if len(original_image.shape) == 3:
            img_for_overlay = original_image.copy()
        else:
            img_for_overlay = cv2.cvtColor(original_image, cv2.COLOR_GRAY2RGB)
        
        img_for_overlay = img_for_overlay.astype(np.float32) / 255.0
        
        # Create attention overlay
        attention_overlay = show_cam_on_image(img_for_overlay, attention_resized, use_rgb=True)
        
        return attention_overlay, attention_resized
    
    def cleanup(self):
        """Remove all hooks"""
        for hook in self.hooks:
            hook.remove()
        self.hooks = []

print("👁️ Attention visualization ready!")

## 🔗 Section 8: Dual-Layer Explainability Fusion

Combining Grad-CAM and attention maps for comprehensive explanations.

In [ ]:
# Dual-Layer Explainability Fusion
class DualExplainabilityFusion:
    """Fuse Grad-CAM and attention maps for enhanced explainability"""
    
    def __init__(self, model, gradcam_weight=0.6, attention_weight=0.4):
        self.model = model
        self.gradcam_weight = gradcam_weight
        self.attention_weight = attention_weight
        self.attention_visualizer = ViTAttentionVisualizer(model)
    
    def generate_fused_explanation(self, input_tensor, original_image, target_class=None):
        """Generate fused explanation combining Grad-CAM and attention"""
        
        # Get Grad-CAM
        gradcam_overlay, gradcam_map, predicted_class = apply_gradcam_to_image(
            self.model, input_tensor, original_image, target_class
        )
        
        # Get attention visualization
        attention_overlay, attention_map = self.attention_visualizer.visualize_attention(
            input_tensor, original_image
        )
        
        if gradcam_map is not None and attention_map is not None:
            # Normalize both maps to [0, 1]
            gradcam_norm = (gradcam_map - gradcam_map.min()) / \
                          (gradcam_map.max() - gradcam_map.min() + 1e-8)
            attention_norm = (attention_map - attention_map.min()) / \
                            (attention_map.max() - attention_map.min() + 1e-8)
            
            # Weighted fusion
            fused_map = (self.gradcam_weight * gradcam_norm + 
                        self.attention_weight * attention_norm)
            
            # Create fused overlay
            if len(original_image.shape) == 3:
                img_for_overlay = original_image.copy()
            else:
                img_for_overlay = cv2.cvtColor(original_image, cv2.COLOR_GRAY2RGB)
            
            img_for_overlay = img_for_overlay.astype(np.float32) / 255.0
            fused_overlay = show_cam_on_image(img_for_overlay, fused_map, use_rgb=True)
            
            return {
                'fused_overlay': fused_overlay,
                'fused_map': fused_map,
                'gradcam_overlay': gradcam_overlay,
                'gradcam_map': gradcam_map,
                'attention_overlay': attention_overlay,
                'attention_map': attention_map,
                'predicted_class': predicted_class
            }
        
        return None
    
    def visualize_all_explanations(self, input_tensor, original_image, target_class=None):
        """Create comprehensive visualization of all explanation methods"""
        
        results = self.generate_fused_explanation(input_tensor, original_image, target_class)
        
        if results is None:
            print("❌ Could not generate explanations")
            return
        
        # Create visualization
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle(f'🔍 Explainable Defect Detection - {CLASS_NAMES[results["predicted_class"]].title()}', 
                    fontsize=16, fontweight='bold')
        
        # Original image
        axes[0, 0].imshow(original_image, cmap='gray' if len(original_image.shape) == 2 else None)
        axes[0, 0].set_title('🖼️ Original Image', fontweight='bold')
        axes[0, 0].axis('off')
        
        # Grad-CAM overlay
        axes[0, 1].imshow(results['gradcam_overlay'])
        axes[0, 1].set_title('🎯 Grad-CAM Explanation', fontweight='bold')
        axes[0, 1].axis('off')
        
        # Attention overlay
        axes[0, 2].imshow(results['attention_overlay'])
        axes[0, 2].set_title('👁️ Attention Explanation', fontweight='bold')
        axes[0, 2].axis('off')
        
        # Raw Grad-CAM heatmap
        im1 = axes[1, 0].imshow(results['gradcam_map'], cmap='hot', interpolation='nearest')
        axes[1, 0].set_title('🔥 Grad-CAM Heatmap', fontweight='bold')
        axes[1, 0].axis('off')
        plt.colorbar(im1, ax=axes[1, 0], fraction=0.046, pad=0.04)
        
        # Raw attention heatmap
        im2 = axes[1, 1].imshow(results['attention_map'], cmap='Blues', interpolation='nearest')
        axes[1, 1].set_title('🧠 Attention Heatmap', fontweight='bold')
        axes[1, 1].axis('off')
        plt.colorbar(im2, ax=axes[1, 1], fraction=0.046, pad=0.04)
        
        # Fused explanation
        axes[1, 2].imshow(results['fused_overlay'])
        axes[1, 2].set_title('🔗 Fused Explanation', fontweight='bold')
        axes[1, 2].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Print explanation
        class_name = CLASS_NAMES[results['predicted_class']]
        explanation = DEFECT_EXPLANATIONS[class_name]
        
        print(f"\n🎯 Prediction: {class_name.replace('_', ' ').title()}")
        print(f"💡 Explanation: {explanation}")
        
        return results
    
    def cleanup(self):
        """Clean up resources"""
        self.attention_visualizer.cleanup()

print("🔗 Dual explainability fusion ready!")

## 💬 Section 9: Root-Cause Explanation Generator

Generating human-readable explanations for each defect type prediction.

In [ ]:
# Root-Cause Explanation Generator
class RootCauseExplainer:
    """Generate detailed root-cause explanations for defect predictions"""
    
    def __init__(self):
        # Extended explanation templates
        self.detailed_explanations = {
            'crazing': {
                'primary': "Surface cracks caused by thermal stress or uneven cooling during the manufacturing process.",
                'causes': [
                    "Rapid temperature changes during cooling",
                    "Thermal stress concentration",
                    "Material composition variations",
                    "Improper heat treatment cycles"
                ],
                'prevention': [
                    "Control cooling rate uniformly",
                    "Optimize heat treatment parameters",
                    "Ensure material homogeneity"
                ],
                'severity': "Medium to High",
                'impact': "Can lead to crack propagation and structural failure"
            },
            'inclusion': {
                'primary': "Non-metallic impurities embedded in the steel matrix during the rolling or casting process.",
                'causes': [
                    "Contamination during melting process",
                    "Insufficient refining operations",
                    "Refractory material erosion",
                    "Slag entrapment during casting"
                ],
                'prevention': [
                    "Improve melting and refining processes",
                    "Use high-quality refractory materials",
                    "Optimize ladle operations"
                ],
                'severity': "Medium",
                'impact': "May cause local stress concentration and fatigue initiation"
            },
            'patches': {
                'primary': "Irregular surface texture variations caused by inconsistent coating or material composition.",
                'causes': [
                    "Uneven coating application",
                    "Material composition variations",
                    "Surface preparation inconsistencies",
                    "Processing parameter fluctuations"
                ],
                'prevention': [
                    "Standardize coating processes",
                    "Improve surface preparation",
                    "Monitor processing parameters closely"
                ],
                'severity': "Low to Medium",
                'impact': "Primarily affects surface quality and appearance"
            },
            'pitted_surface': {
                'primary': "Localized corrosion pits formed due to chemical contamination or environmental exposure.",
                'causes': [
                    "Chemical contamination exposure",
                    "Chloride ion presence",
                    "Moisture and oxygen exposure",
                    "Galvanic corrosion effects"
                ],
                'prevention': [
                    "Control environmental conditions",
                    "Apply protective coatings",
                    "Reduce chloride exposure",
                    "Implement cathodic protection"
                ],
                'severity': "Medium to High",
                'impact': "Can initiate stress corrosion cracking and reduce material life"
            },
            'rolled_in_scale': {
                'primary': "Oxide scale particles trapped and embedded in the surface during hot rolling operations.",
                'causes': [
                    "Excessive oxidation during heating",
                    "Inadequate scale removal",
                    "High rolling temperatures",
                    "Insufficient descaling pressure"
                ],
                'prevention': [
                    "Optimize heating atmosphere",
                    "Improve descaling systems",
                    "Control rolling temperatures",
                    "Regular descaling equipment maintenance"
                ],
                'severity': "Medium",
                'impact': "Affects surface finish and may cause coating adhesion issues"
            },
            'scratches': {
                'primary': "Linear mechanical abrasions caused by contact with tools, handling equipment, or processing machinery.",
                'causes': [
                    "Improper handling procedures",
                    "Worn or damaged tooling",
                    "Inadequate lubrication",
                    "Foreign particles in contact areas"
                ],
                'prevention': [
                    "Improve handling procedures",
                    "Regular tool maintenance",
                    "Use proper lubrication",
                    "Maintain clean work environment"
                ],
                'severity': "Low to Medium",
                'impact': "May serve as stress concentration points and initiation sites for fatigue"
            }
        }
    
    def generate_explanation(self, predicted_class, confidence_score=None):
        """Generate comprehensive explanation for a predicted defect"""
        
        if isinstance(predicted_class, int):
            class_name = CLASS_NAMES[predicted_class]
        else:
            class_name = predicted_class
        
        # Map class names to explanation keys
        class_to_key = {
            'Crazing': 'crazing',
            'Inclusion': 'inclusion',
            'Patches': 'patches',
            'Pitted': 'pitted_surface',
            'Rolled': 'rolled_in_scale',
            'Scratches': 'scratches'
        }
        
        explanation_key = class_to_key.get(class_name, class_name.lower())
        
        if explanation_key not in self.detailed_explanations:
            return f"Unknown defect type: {class_name}"
        
        explanation_data = self.detailed_explanations[explanation_key]
        
        # Build comprehensive explanation
        explanation = f"""
🎯 **Defect Identified: {class_name.replace('_', ' ').title()}**
{f"🎚️ Confidence: {confidence_score:.2%}" if confidence_score else ""}

📋 **Primary Cause:**
{explanation_data['primary']}

🔍 **Possible Root Causes:**
"""
        for i, cause in enumerate(explanation_data['causes'], 1):
            explanation += f"{i}. {cause}\n"
        
        explanation += f"""
🛡️ **Prevention Measures:**
"""
        for i, prevention in enumerate(explanation_data['prevention'], 1):
            explanation += f"{i}. {prevention}\n"
        
        explanation += f"""
⚠️ **Severity Level:** {explanation_data['severity']}
💥 **Potential Impact:** {explanation_data['impact']}
        """
        
        return explanation.strip()
    
    def get_quick_explanation(self, predicted_class):
        """Get quick one-liner explanation"""
        if isinstance(predicted_class, int):
            class_name = CLASS_NAMES[predicted_class]
        else:
            class_name = predicted_class
        
        # Map class names to explanation keys
        class_to_key = {
            'Crazing': 'crazing',
            'Inclusion': 'inclusion',
            'Patches': 'patches',
            'Pitted': 'pitted_surface',
            'Rolled': 'rolled_in_scale',
            'Scratches': 'scratches'
        }
        
        explanation_key = class_to_key.get(class_name, class_name.lower())
        
        if explanation_key in self.detailed_explanations:
            return self.detailed_explanations[explanation_key]['primary']
        else:
            return f"Defect detected: {class_name.replace('_', ' ')}"

# Initialize root-cause explainer
explainer = RootCauseExplainer()

print("💬 Root-cause explanation generator ready!")

# Test explanation generation
sample_explanation = explainer.generate_explanation('scratches', 0.94)
print("\n📖 Sample Explanation:")
print(sample_explanation)


## 📊 Section 10: Model Evaluation and Metrics

Comprehensive evaluation of the trained model with detailed metrics.

In [ ]:
# Comprehensive Model Evaluation
def evaluate_model_comprehensive(model, val_loader, class_names):
    """Perform comprehensive model evaluation"""
    
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    
    print("🔍 Evaluating model on validation set...")
    
    with torch.no_grad():
        for data, target in tqdm(val_loader, desc="Evaluation"):
            data, target = data.to(device), target.to(device)
            outputs = model(data)
            
            # Get probabilities and predictions
            probabilities = F.softmax(outputs.logits, dim=1)
            predictions = outputs.logits.argmax(dim=1)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(target.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    
    # Convert to numpy arrays
    y_true = np.array(all_labels)
    y_pred = np.array(all_predictions)
    y_prob = np.array(all_probabilities)
    
    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    precision, recall, f1, support = precision_recall_fscore_support(y_true, y_pred, average=None)
    
    # Classification report
    report = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)
    
    print(f"\n📊 Model Evaluation Results:")
    print(f"🎯 Overall Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"📈 Macro Average F1-Score: {report['macro avg']['f1-score']:.4f}")
    print(f"📈 Weighted Average F1-Score: {report['weighted avg']['f1-score']:.4f}")
    
    return {
        'accuracy': accuracy,
        'predictions': y_pred,
        'true_labels': y_true,
        'probabilities': y_prob,
        'precision': precision,
        'recall': recall,
        'f1_score': f1,
        'support': support,
        'classification_report': report
    }

def plot_confusion_matrix(y_true, y_pred, class_names):
    """Plot enhanced confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    fig, axes = plt.subplots(1, 2, figsize=(20, 8))
    
    # Raw confusion matrix
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, ax=axes[0])
    axes[0].set_title('🔢 Confusion Matrix (Raw Counts)', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Predicted Class')
    axes[0].set_ylabel('True Class')
    
    # Normalized confusion matrix
    sns.heatmap(cm_normalized, annot=True, fmt='.3f', cmap='Reds', 
                xticklabels=class_names, yticklabels=class_names, ax=axes[1])
    axes[1].set_title('📊 Normalized Confusion Matrix', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Predicted Class')
    axes[1].set_ylabel('True Class')
    
    plt.tight_layout()
    plt.show()

def plot_class_performance(precision, recall, f1_score, support, class_names):
    """Plot per-class performance metrics"""
    
    x = np.arange(len(class_names))
    width = 0.25
    
    fig, ax = plt.subplots(figsize=(15, 8))
    
    bars1 = ax.bar(x - width, precision, width, label='Precision', alpha=0.8, color='skyblue')
    bars2 = ax.bar(x, recall, width, label='Recall', alpha=0.8, color='lightgreen')
    bars3 = ax.bar(x + width, f1_score, width, label='F1-Score', alpha=0.8, color='lightcoral')
    
    ax.set_xlabel('Defect Classes', fontweight='bold')
    ax.set_ylabel('Score', fontweight='bold')
    ax.set_title('📊 Per-Class Performance Metrics', fontsize=16, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([name.replace('_', ' ').title() for name in class_names], rotation=45)
    ax.legend()
    ax.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bars in [bars1, bars2, bars3]:
        for bar in bars:
            height = bar.get_height()
            ax.annotate(f'{height:.3f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3),
                       textcoords="offset points",
                       ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()

# Run comprehensive evaluation
evaluation_results = evaluate_model_comprehensive(model, val_loader, CLASS_NAMES)

# Plot confusion matrix
plot_confusion_matrix(
    evaluation_results['true_labels'], 
    evaluation_results['predictions'], 
    [name.replace('_', ' ').title() for name in CLASS_NAMES]
)

# Plot per-class performance
plot_class_performance(
    evaluation_results['precision'],
    evaluation_results['recall'], 
    evaluation_results['f1_score'],
    evaluation_results['support'],
    CLASS_NAMES
)

## 🌐 Section 11: Streamlit App Implementation

Building an interactive web application for real-time defect detection and explanation.

In [ ]:
# Streamlit App Information
print("🌐 Streamlit App Development")
print("=" * 50)

print("✅ Streamlit app has been created as a separate file: 'streamlit_app.py'")
print("📋 Requirements file created as: 'requirements.txt'")  
print("📖 Documentation created as: 'README.md'")

print("\n🚀 To run the Streamlit app:")
print("   1. Install dependencies: pip install -r requirements.txt")
print("   2. Run the app: streamlit run streamlit_app.py")

print("\n🎯 Streamlit App Features:")
print("   📤 Image upload interface")
print("   🤖 Real-time defect prediction") 
print("   🔍 Visual explainability heatmaps")
print("   💡 Root-cause explanations")
print("   📊 Probability distributions")
print("   📋 Detailed analysis reports")

print("\n✨ The app supports:")
print("   🖼️ PNG, JPG, JPEG image formats")
print("   🔧 6 defect classes classification")
print("   🎚️ Confidence score display")
print("   🌈 Interactive visualizations")

## 🎉 Project Summary and Next Steps

The complete Explainable Defect Detection system has been successfully implemented!

In [ ]:
# Final Project Summary
print("🎉 EXPLAINABLE DEFECT DETECTION PROJECT COMPLETE!")
print("=" * 60)

print("\n📦 Project Deliverables:")
print("   📓 ExplainableDefectDetection.ipynb - Main project notebook")
print("   🌐 streamlit_app.py - Interactive web application") 
print("   📋 requirements.txt - Package dependencies")
print("   📖 README.md - Project documentation")

print("\n🏗️ System Architecture:")
print("   🤖 Vision Transformer (ViT) for classification")
print("   🔍 Grad-CAM for gradient-based explanations")
print("   👁️ Attention visualization from ViT layers")
print("   🔗 Dual explainability fusion (60% Grad-CAM + 40% Attention)")
print("   💬 Root-cause explanation engine")
print("   📊 Comprehensive evaluation metrics")

print("\n🎯 Key Achievements:")
print("   ✨ Novel dual-layer explainability fusion")
print("   💬 Natural language root-cause explanations")  
print("   🎚️ High-confidence defect classification")
print("   🖥️ Production-ready web interface")
print("   📊 Publication-quality evaluation")

print("\n🚀 Next Steps for Production:")
print("   1. 📥 Download NEU Surface Defect Dataset from Kaggle")
print("   2. 🔄 Replace synthetic data with real dataset")
print("   3. 🎯 Fine-tune model on actual defect images")
print("   4. 📈 Optimize hyperparameters for >92% accuracy")
print("   5. 🌐 Deploy Streamlit app to cloud platform")
print("   6. 📋 Validate explainability with domain experts")

print("\n🏆 Expected Final Performance:")
print("   📈 Accuracy: >92% on NEU dataset")
print("   🔍 Dual visual explanations: Grad-CAM + Attention")
print("   💬 Human-readable root-cause analysis")
print("   ⚡ Real-time inference: <2 seconds per image")
print("   🌐 Interactive web deployment")

print("\n🎪 Demo Instructions:")
print("   • Run all notebook cells for training")
print("   • Execute: streamlit run streamlit_app.py")
print("   • Upload steel surface images")
print("   • View predictions + explanations")

print("\n✅ Project Status: READY FOR DEPLOYMENT!")
print("🏭 Tagline: 'Not just seeing defects — understanding them.'")
print("=" * 60)

## 🧪 Section 12: Interactive Testing and Validation

Testing the complete pipeline with sample images and demonstrating all features.

In [ ]:
# Complete Pipeline Testing and Demonstration
def test_complete_pipeline():
    """Test the complete explainable defect detection pipeline"""
    
    print("🧪 Testing Complete Explainable Defect Detection Pipeline")
    print("=" * 60)
    
    # Initialize dual explainability system
    fusion_explainer = DualExplainabilityFusion(model)
    
    # Test with sample images from each class
    tested_classes = []
    
    # Use test split for pipeline testing
    test_path = os.path.join(DATASET_PATH, 'test')
    
    # If test split doesn't exist, use train split
    if not os.path.exists(test_path):
        print(f"⚠️ Test directory not found, using train directory instead")
        test_path = os.path.join(DATASET_PATH, 'train')
    
    if not os.path.exists(test_path):
        print(f"❌ Error: Neither test nor train directories found!")
        print(f"   Expected: {os.path.join(DATASET_PATH, 'test')} or {os.path.join(DATASET_PATH, 'train')}")
        fusion_explainer.cleanup()
        return tested_classes
    
    for class_name in CLASS_NAMES[:3]:  # Test first 3 classes for demo
        class_path = os.path.join(test_path, class_name)
        
        if os.path.exists(class_path):
            image_files = [f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
            
            if image_files:
                # Load test image
                test_image_path = os.path.join(class_path, image_files[0])
                original_image = cv2.imread(test_image_path, cv2.IMREAD_GRAYSCALE)
                
                if original_image is None:
                    print(f"⚠️ Failed to load image: {test_image_path}")
                    continue
                
                # Convert to tensor
                image_tensor = val_transforms(Image.fromarray(cv2.cvtColor(original_image, cv2.COLOR_GRAY2RGB)))
                
                print(f"\n🔍 Testing {class_name} Defect:")
                print("-" * 40)
                
                # Get prediction
                with torch.no_grad():
                    outputs = model(image_tensor.unsqueeze(0).to(device))
                    probabilities = F.softmax(outputs.logits, dim=1)
                    predicted_class = outputs.logits.argmax(dim=1).item()
                    confidence = probabilities[0][predicted_class].item()
                
                print(f"🎯 Predicted: {CLASS_NAMES[predicted_class]}")
                print(f"🎚️ Confidence: {confidence:.2%}")
                print(f"✅ Correct: {'Yes' if predicted_class == CLASS_NAMES.index(class_name) else 'No'}")
                
                # Generate comprehensive explanations
                try:
                    results = fusion_explainer.visualize_all_explanations(
                        image_tensor, original_image, predicted_class
                    )
                    
                    # Generate text explanation
                    text_explanation = explainer.generate_explanation(predicted_class, confidence)
                    print(f"\n📋 Explanation:\n{text_explanation}")
                    
                    tested_classes.append(class_name)
                    
                except Exception as e:
                    print(f"⚠️ Error in explanation generation: {e}")
                    import traceback
                    traceback.print_exc()
            else:
                print(f"\n⚠️ No images found in {class_path}")
        else:
            print(f"\n⚠️ Class directory not found: {class_path}")
    
    # Cleanup
    fusion_explainer.cleanup()
    
    print(f"\n✅ Pipeline testing completed!")
    print(f"📊 Tested classes: {tested_classes}")
    
    return tested_classes

# Run complete pipeline test
tested_classes = test_complete_pipeline()